In [16]:
# | eval: false

In [17]:
#| default_exp data_loaders

In [18]:
#|export
import pandas as pd
import yfinance as yf
import torch
import numpy as np
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset, DataLoader
from arch import arch_model

In [19]:
#| export
def get_nifty_regime_data(start_date="2015-01-01", end_date="2026-04-01"):
    nifty    = yf.download("^NSEI",     start=start_date, end=end_date, auto_adjust=True)
    vix_data = yf.download("^INDIAVIX", start=start_date, end=end_date, auto_adjust=True)
    if isinstance(nifty.columns, pd.MultiIndex):
        df  = pd.DataFrame({"Close": nifty["Close"].iloc[:, 0],
                             "Volume": nifty["Volume"].iloc[:, 0]})
        vix = vix_data["Close"].iloc[:, 0]
    else:
        df  = nifty[["Close", "Volume"]].copy()
        vix = vix_data["Close"]

    df["log_return"] = np.log(df["Close"] / df["Close"].shift(1))
    for lag in [1, 2, 3, 5, 10, 21]:
        df[f"return_lag_{lag}"] = df["log_return"].shift(lag)
    for w in [5, 10, 21, 63]:
        df[f"cum_return_{w}d"] = df["log_return"].rolling(w).sum().shift(1)
    for w in [5, 10, 21, 63]:
        df[f"rvol_{w}d"] = df["log_return"].rolling(w).std().shift(1) * np.sqrt(252)
    df["vol_ratio_5_21"]  = df["rvol_5d"]  / df["rvol_21d"]
    df["vol_ratio_21_63"] = df["rvol_21d"] / df["rvol_63d"]
    df["realized_vol"]    = df["rvol_21d"]

    idx = df.index
    df["dow_sin"]     = np.sin(2 * np.pi * idx.dayofweek / 5)
    df["dow_cos"]     = np.cos(2 * np.pi * idx.dayofweek / 5)
    df["month_sin"]   = np.sin(2 * np.pi * (idx.month - 1) / 12)
    df["month_cos"]   = np.cos(2 * np.pi * (idx.month - 1) / 12)
    df["dom_sin"]     = np.sin(2 * np.pi * (idx.day - 1) / 31)
    df["dom_cos"]     = np.cos(2 * np.pi * (idx.day - 1) / 31)
    df["quarter_sin"] = np.sin(2 * np.pi * (idx.quarter - 1) / 4)
    df["quarter_cos"] = np.cos(2 * np.pi * (idx.quarter - 1) / 4)

    df["skew"]      = df["log_return"].rolling(60).skew().shift(1)
    df["kurtosis"]  = df["log_return"].rolling(60).kurt().shift(1)
    df["vol_spike"] = df["Volume"] / df["Volume"].rolling(20).mean().shift(1)
    delta = df["Close"].diff()
    gain  = delta.where(delta > 0, 0).rolling(14).mean().shift(1)
    loss  = (-delta.where(delta < 0, 0)).rolling(14).mean().shift(1)
    df["RSI"]          = 100 - (100 / (1 + gain / loss))
    df["RSI_velocity"] = df["RSI"].diff(3)
    df["sma_200"]      = df["Close"].rolling(200).mean().shift(1)

    df["regime"] = 0
    df.loc[
        (df["Close"].shift(1) < df["sma_200"]) |
        ((df["skew"] < -0.5) & (df["realized_vol"] > df["realized_vol"].rolling(100).mean().shift(1))),
        "regime"
    ] = 1

    df["target_return"] = df["log_return"].shift(-1)
    vix_aligned = pd.Series(vix, name="VIX").reindex(df.index, method="ffill")
    df["VIX"] = vix_aligned.shift(1)
    df = df.dropna()

    past_return_cols = ([f"return_lag_{l}" for l in [1, 2, 3, 5, 10, 21]] +
                        [f"cum_return_{w}d" for w in [5, 10, 21, 63]])
    rvol_cols        = ([f"rvol_{w}d" for w in [5, 10, 21, 63]] +
                        ["vol_ratio_5_21", "vol_ratio_21_63"])
    time_cols        = ["dow_sin", "dow_cos", "month_sin", "month_cos",
                        "dom_sin", "dom_cos", "quarter_sin", "quarter_cos"]
    original_cols    = ["realized_vol", "VIX", "RSI", "skew", "kurtosis",
                        "vol_spike", "RSI_velocity", "sma_200", "Close", "regime"]
    cols = ["target_return"] + past_return_cols + rvol_cols + time_cols + original_cols
    return df[cols]

risk_df = get_nifty_regime_data()
print(f"Shape: {risk_df.shape}")
print(f"\nRegime counts:\n{risk_df['regime'].value_counts()}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Shape: (2566, 35)

Regime counts:
regime
0    1787
1     779
Name: count, dtype: int64


In [20]:
#| export
def compute_regime_score(df):
    sma_distance = (df["sma_200"] - df["Close"].shift(1)) / df["sma_200"]
    signal_1 = sma_distance.clip(0, 0.15) / 0.15
    signal_2 = df["realized_vol"].rolling(252).rank(pct=True).shift(1).fillna(0.5)
    signal_3 = (-df["skew"]).clip(0, 2) / 2
    signal_4 = df["VIX"].rolling(252).rank(pct=True).shift(1).fillna(0.5)
    return (0.30 * signal_1 + 0.30 * signal_2 +
            0.20 * signal_3 + 0.20 * signal_4).clip(0, 1)


In [21]:
risk_df = get_nifty_regime_data()
risk_df['regime_score'] = compute_regime_score(risk_df)
risk_df['regime_score'] = risk_df['regime_score'].fillna(0.5)
risk_df['regime'] = (risk_df['regime_score'] > 0.5).astype(int)
risk_df.drop(columns=['sma_200','Close'], inplace=True)
risk_df.head()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,target_return,return_lag_1,return_lag_2,return_lag_3,return_lag_5,return_lag_10,return_lag_21,cum_return_5d,cum_return_10d,cum_return_21d,...,quarter_cos,realized_vol,VIX,RSI,skew,kurtosis,vol_spike,RSI_velocity,regime,regime_score
Date,,,,,,,,,,,,,,,,,,,,,
2015-10-27,-0.007523,-0.004216,0.005288,-0.001205,0.004469,-0.005645,0.004336,0.002715,0.008614,0.055830,...,-1.836970e-16,0.110922,17.230000,63.648657,-1.867438,7.837366,0.923993,-16.027928,0,0.500000
2015-10-28,-0.007302,-0.003353,-0.004216,0.005288,-0.001621,-0.001462,0.002870,-0.005107,0.010906,0.048141,...,-1.836970e-16,0.112642,16.540001,57.820192,-1.840251,7.784645,1.129109,-17.032109,0,0.467612
2015-10-29,-0.005681,-0.007523,-0.003353,-0.004216,-0.001205,-0.002931,-0.009295,-0.011009,0.004846,0.037748,...,-1.836970e-16,0.117614,17.040001,49.435055,-1.787560,7.617661,1.279939,-27.030146,0,0.476838
2015-10-30,-0.001861,-0.007302,-0.007523,-0.003353,0.005288,0.008792,0.006087,-0.017106,0.000475,0.039741,...,-1.836970e-16,0.115428,17.580000,48.428844,-1.770332,7.721606,1.182082,-15.219813,0,0.489299
2015-11-02,0.001229,-0.005681,-0.007302,-0.007523,-0.004216,0.007145,0.013374,-0.028074,-0.013998,0.027973,...,-1.836970e-16,0.117224,17.879999,38.647577,-1.738055,7.611434,0.809108,-19.172615,0,0.497006


In [22]:
#| export
def fit_garch_volatility(risk_df):
    n_total   = len(risk_df)
    train_end = int(0.80 * n_total)

    train_returns_pct = risk_df["target_return"].iloc[:train_end] * 100
    print("Fitting GARCH(1,1) on training data...")
    garch_spec = arch_model(y=train_returns_pct, mean="Constant", vol="Garch",
                            p=1, q=1, dist="Normal")
    garch_fit  = garch_spec.fit(disp="off")
    print(garch_fit.summary())

    all_returns_pct = risk_df["target_return"] * 100
    garch_full      = arch_model(y=all_returns_pct, mean="Constant", vol="Garch",
                                p=1, q=1, dist="Normal")
    garch_fixed     = garch_full.fix(garch_fit.params)
    cond_vol_pct    = garch_fixed.conditional_volatility

    risk_df["garch_vol"]   = (cond_vol_pct / 100.0).shift(1)
    risk_df["garch_vol"]   = risk_df["garch_vol"].fillna(risk_df["realized_vol"] / np.sqrt(252))
    risk_df["garch_resid"] = (risk_df["target_return"] / risk_df["garch_vol"]).clip(-10, 10)
    return risk_df

In [23]:
risk_df = fit_garch_volatility(risk_df)
print(f"\nGARCH residual stats:")
print(f"  mean={risk_df['garch_resid'].mean():.4f}")
print(f"  std ={risk_df['garch_resid'].std():.4f}  (should be ~1.0)")
print(f"  5th ={risk_df['garch_resid'].quantile(0.05):.4f}")
print(f"  1st ={risk_df['garch_resid'].quantile(0.01):.4f}")

Fitting GARCH(1,1) on training data...
                     Constant Mean - GARCH Model Results                      
Dep. Variable:          target_return   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -2619.48
Distribution:                  Normal   AIC:                           5246.95
Method:            Maximum Likelihood   BIC:                           5269.46
                                        No. Observations:                 2052
Date:                Fri, Apr 24 2026   Df Residuals:                     2051
Time:                        08:51:35   Df Model:                            1
                                Mean Model                                
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
mu             0.0776  1.

In [24]:
#| export
PAST_RETURN_COLS   = ([f"return_lag_{l}" for l in [1, 2, 3, 5, 10, 21]] +
                      [f"cum_return_{w}d" for w in [5, 10, 21, 63]])
RVOL_COLS          = ([f"rvol_{w}d" for w in [5, 10, 21, 63]] +
                      ["vol_ratio_5_21", "vol_ratio_21_63"])
ORIGINAL_CONT_COLS = ["realized_vol", "VIX", "RSI", "skew",
                       "kurtosis", "vol_spike", "RSI_velocity", "garch_vol"]
UNSCALED_CONT_COLS = ["regime_score",
                       "dow_sin", "dow_cos", "month_sin", "month_cos",
                       "dom_sin", "dom_cos", "quarter_sin", "quarter_cos"]
SCALED_CONT_COLS   = ORIGINAL_CONT_COLS + PAST_RETURN_COLS + RVOL_COLS

MARKET_DIM  = len(SCALED_CONT_COLS) + 1
TIME_DIM    = len(UNSCALED_CONT_COLS) - 1
CONTEXT_DIM = len(SCALED_CONT_COLS) + len(UNSCALED_CONT_COLS)
print(f"MARKET_DIM={MARKET_DIM}, TIME_DIM={TIME_DIM}, CONTEXT_DIM={CONTEXT_DIM}")


class NiftyRiskDataset(Dataset):
    def __init__(self, df):
        self.x0             = df["garch_resid"].values.astype(np.float32)
        self.context_cat    = df[["regime"]].values.astype("int64")
        self.context_scaled = df[SCALED_CONT_COLS].values.astype(np.float32)
        self.context_fixed  = df[UNSCALED_CONT_COLS].values.astype(np.float32)

    def __len__(self): return len(self.x0)

    def __getitem__(self, idx):
        cont = np.concatenate([self.context_scaled[idx], self.context_fixed[idx]])
        return (
            torch.tensor([self.x0[idx]]),
            torch.tensor(self.context_cat[idx][0]) + 1,
            torch.from_numpy(cont),
        )



def prepare_dataloaders(df, batch_size=128, train_size=0.80, val_size=0.10):
    n         = len(df)
    train_end = int(train_size * n)
    val_end   = int((train_size + val_size) * n)
    train_orig = df[:train_end].copy()
    val_orig   = df[train_end:val_end].copy()
    cond_orig  = df[val_end:].copy()
    print(f"Train: {len(train_orig):,}  Val: {len(val_orig):,}  Test: {len(cond_orig):,}")

    scaler_x    = StandardScaler().fit(train_orig[["garch_resid"]])
    scaler_cond = StandardScaler().fit(train_orig[SCALED_CONT_COLS])

    def apply_scalers(src):
        dst = src.copy()
        dst[["garch_resid"]]  = scaler_x.transform(src[["garch_resid"]])
        dst[SCALED_CONT_COLS] = scaler_cond.transform(src[SCALED_CONT_COLS])
        return dst

    train_dl = DataLoader(NiftyRiskDataset(apply_scalers(train_orig)),
                          batch_size=batch_size, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(NiftyRiskDataset(apply_scalers(val_orig)),
                          batch_size=batch_size, shuffle=False, num_workers=0)
    return train_dl, val_dl, cond_orig, scaler_x, scaler_cond, val_orig


MARKET_DIM=25, TIME_DIM=8, CONTEXT_DIM=33


In [25]:
train_dl, val_dl, condition_df, scaler_x, scaler_cond, val_df = prepare_dataloaders(risk_df)
print(f"\nscaler_x mean={scaler_x.mean_[0]:.4f}  scale={scaler_x.scale_[0]:.4f}  (mean~0, scale~1)")

print(f"\nDataLoader batch info:")
print(f"  Train batches: {len(train_dl)}")
print(f"  Validation batches: {len(val_dl)}")
print(f"\nConditioning dataframe shape: {condition_df.shape}")
print(f"Conditioning date range: {condition_df.index[0]} to {condition_df.index[-1]}")

Train: 2,052  Val: 257  Test: 257

scaler_x mean=0.0591  scale=1.0063  (mean~0, scale~1)

DataLoader batch info:
  Train batches: 17
  Validation batches: 3

Conditioning dataframe shape: (257, 36)
Conditioning date range: 2025-03-12 00:00:00 to 2026-03-27 00:00:00
